# Art Style EDA and Training
Exploratory Data Analysis and Model Training for 8 Art Styles Classification using images from Google Drive.

# Required Libraries

In [3]:
# Install required packages (run this first!)
import subprocess
import sys

packages = [
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'pillow',
    'scikit-learn',
    'opencv-python',
    'tensorflow',
    'keras'
]

print("Installing required packages...")
for package in packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
        print(f"✓ {package} installed")
    except Exception as e:
        print(f"! {package} - {e}")

print("\nAll packages installed successfully!")

Installing required packages...
✓ pandas installed
✓ pandas installed
✓ numpy installed
✓ numpy installed
✓ matplotlib installed
✓ matplotlib installed
✓ seaborn installed
✓ seaborn installed
✓ pillow installed
✓ pillow installed
✓ scikit-learn installed
✓ scikit-learn installed
✓ opencv-python installed
✓ opencv-python installed
✓ tensorflow installed
✓ tensorflow installed
✓ keras installed

All packages installed successfully!
✓ keras installed

All packages installed successfully!


In [4]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import defaultdict, Counter
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# For Google Colab (uncomment if running in Colab)
# from google.colab import drive
# drive.mount('/content/drive')

# Set style for visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

Libraries imported successfully!


# Mount Google Drive and Load Data

In [ ]:
# Uncomment the following lines if running in Google Colab
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_PATH = '/content/drive/MyDrive/ArtStyles'  # Update with your folder path

# For local testing, define the local path to your art styles folder
# Adjust this path to where your art styles images are stored locally
DRIVE_PATH = 'My Drive/ Paintings'  # Change this to your local folder or Colab path

# Define art styles
ART_STYLES = ['art_nouveau', 'cubism', 'expressionism', 'fauvism', 
              'futurism', 'impressionism', 'neoclassicism', 'surrealism']

# Load image data from folders
image_data = []
for style in ART_STYLES:
    style_path = os.path.join(DRIVE_PATH, style)
    if os.path.exists(style_path):
        for img_file in os.listdir(style_path):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp')):
                image_data.append({
                    'file_name': img_file,
                    'style': style,
                    'file_path': os.path.join(style_path, img_file),
                    'file_size': os.path.getsize(os.path.join(style_path, img_file)) / 1024  # KB
                })
    else:
        print(f"Warning: {style_path} not found. Create this folder and add images.")

# Create DataFrame
df_images = pd.DataFrame(image_data)

if len(df_images) > 0:
    print(f"Total images loaded: {len(df_images)}")
    print(f"\nDataset structure:\n{df_images.head()}")
else:
    print("No images found. Please ensure your art styles folders are set up correctly.")

No images found. Please ensure your art styles folders are set up correctly.


# Dataset Overview and Statistics

In [ ]:
if len(df_images) > 0:
    print("=" * 60)
    print("DATASET OVERVIEW AND STATISTICS")
    print("=" * 60)
    print(f"\nTotal number of images: {len(df_images)}")
    print(f"Number of art styles: {df_images['style'].nunique()}")
    print(f"Total dataset size: {df_images['file_size'].sum():.2f} KB ({df_images['file_size'].sum() / 1024:.2f} MB)")
    
    print("\n" + "="*60)
    print("Images per Art Style:")
    print("="*60)
    style_counts = df_images['style'].value_counts()
    for style, count in style_counts.items():
        print(f"{style.upper()}: {count} images")
    
    print("\n" + "="*60)
    print("File Size Statistics (KB):")
    print("="*60)
    print(f"Mean: {df_images['file_size'].mean():.2f} KB")
    print(f"Median: {df_images['file_size'].median():.2f} KB")
    print(f"Min: {df_images['file_size'].min():.2f} KB")
    print(f"Max: {df_images['file_size'].max():.2f} KB")
    print(f"Std Dev: {df_images['file_size'].std():.2f} KB")
else:
    print("No image data available for analysis.")

## Section 4: Image Dimension Analysis

In [ ]:
# Analyze image dimensions
dimensions_data = []
for idx, row in df_images.iterrows():
    try:
        img = Image.open(row['file_path'])
        width, height = img.size
        aspect_ratio = width / height if height > 0 else 0
        dimensions_data.append({
            'style': row['style'],
            'width': width,
            'height': height,
            'aspect_ratio': aspect_ratio
        })
    except Exception as e:
        print(f"Error processing {row['file_path']}: {e}")

df_dimensions = pd.DataFrame(dimensions_data)

if len(df_dimensions) > 0:
    print("Image Dimension Statistics:")
    print(df_dimensions.groupby('style')[['width', 'height', 'aspect_ratio']].describe().round(2))
    
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Width distribution
    df_dimensions.boxplot(column='width', by='style', ax=axes[0, 0])
    axes[0, 0].set_title('Image Width Distribution by Style')
    axes[0, 0].set_xlabel('Art Style')
    axes[0, 0].set_ylabel('Width (pixels)')
    
    # Height distribution
    df_dimensions.boxplot(column='height', by='style', ax=axes[0, 1])
    axes[0, 1].set_title('Image Height Distribution by Style')
    axes[0, 1].set_xlabel('Art Style')
    axes[0, 1].set_ylabel('Height (pixels)')
    
    # Aspect ratio distribution
    df_dimensions.boxplot(column='aspect_ratio', by='style', ax=axes[1, 0])
    axes[1, 0].set_title('Image Aspect Ratio Distribution by Style')
    axes[1, 0].set_xlabel('Art Style')
    axes[1, 0].set_ylabel('Aspect Ratio')
    
    # Scatter plot: Width vs Height
    for style in df_dimensions['style'].unique():
        style_data = df_dimensions[df_dimensions['style'] == style]
        axes[1, 1].scatter(style_data['width'], style_data['height'], label=style, alpha=0.6)
    axes[1, 1].set_title('Width vs Height')
    axes[1, 1].set_xlabel('Width (pixels)')
    axes[1, 1].set_ylabel('Height (pixels)')
    axes[1, 1].legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()
else:
    print("No valid image dimensions data available.")

## Section 5: Art Style Distribution

In [ ]:
if len(df_images) > 0:
    style_counts = df_images['style'].value_counts()
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Bar plot
    style_counts.plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
    axes[0].set_title('Image Count per Art Style', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Art Style')
    axes[0].set_ylabel('Number of Images')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Pie chart
    colors = plt.cm.Set3(np.linspace(0, 1, len(style_counts)))
    axes[1].pie(style_counts.values, labels=style_counts.index, autopct='%1.1f%%', 
                colors=colors, startangle=90)
    axes[1].set_title('Distribution of Images by Art Style', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\nStyle Distribution Summary:")
    for style, count in style_counts.items():
        percentage = (count / len(df_images)) * 100
        print(f"{style.upper()}: {count} images ({percentage:.1f}%)")
else:
    print("No data available for distribution analysis.")

## Section 6: Sample Images Visualization

In [ ]:
if len(df_images) > 0:
    # Display 2 sample images from each art style
    fig, axes = plt.subplots(len(ART_STYLES), 2, figsize=(12, 4*len(ART_STYLES)))
    
    if len(ART_STYLES) == 1:
        axes = axes.reshape(1, -1)
    
    for row, style in enumerate(ART_STYLES):
        style_images = df_images[df_images['style'] == style]['file_path'].tolist()
        
        for col in range(2):
            if col < len(style_images):
                try:
                    img = Image.open(style_images[col])
                    axes[row, col].imshow(img)
                    axes[row, col].set_title(f'{style.upper()} - Sample {col+1}', fontweight='bold')
                    axes[row, col].axis('off')
                except Exception as e:
                    axes[row, col].text(0.5, 0.5, f'Error loading image', 
                                      ha='center', va='center', transform=axes[row, col].transAxes)
                    axes[row, col].axis('off')
            else:
                axes[row, col].text(0.5, 0.5, 'No image available', 
                                  ha='center', va='center', transform=axes[row, col].transAxes)
                axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No images available to display.")

## Section 7: Image Quality Assessment

In [ ]:
# Check for corrupted images and image formats
corrupted_images = []
image_formats = defaultdict(int)
valid_images = []

for idx, row in df_images.iterrows():
    try:
        img = Image.open(row['file_path'])
        img.verify()  # Verify image integrity
        image_formats[img.format] += 1
        valid_images.append(row['file_path'])
    except Exception as e:
        corrupted_images.append({'file': row['file_path'], 'error': str(e)})

print("=" * 60)
print("IMAGE QUALITY ASSESSMENT")
print("=" * 60)
print(f"\nTotal images analyzed: {len(df_images)}")
print(f"Valid images: {len(valid_images)}")
print(f"Corrupted images: {len(corrupted_images)}")

if len(corrupted_images) > 0:
    print("\nCorrupted Images:")
    for item in corrupted_images:
        print(f"  - {item['file']}: {item['error']}")

print("\nImage Format Distribution:")
for fmt, count in sorted(image_formats.items(), key=lambda x: x[1], reverse=True):
    print(f"  {fmt}: {count} images")

# Visualize format distribution
if len(image_formats) > 0:
    fig, ax = plt.subplots(figsize=(8, 6))
    formats = list(image_formats.keys())
    counts = list(image_formats.values())
    ax.bar(formats, counts, color='coral', edgecolor='black')
    ax.set_title('Image Format Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Image Format')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()

## Section 8: Color Distribution Analysis

In [ ]:
# Analyze color distribution by art style
def analyze_colors(image_path, num_colors=5):
    """Extract dominant colors from an image"""
    try:
        img = Image.open(image_path).convert('RGB')
        img_array = np.array(img)
        pixels = img_array.reshape(-1, 3)
        
        # Calculate average brightness
        brightness = np.mean(img_array)
        
        # Calculate color variance
        color_variance = np.var(img_array, axis=0).mean()
        
        return brightness, color_variance
    except Exception as e:
        return None, None

color_analysis = []
for idx, row in df_images.iterrows():
    brightness, variance = analyze_colors(row['file_path'])
    if brightness is not None:
        color_analysis.append({
            'style': row['style'],
            'brightness': brightness,
            'color_variance': variance
        })

df_colors = pd.DataFrame(color_analysis)

if len(df_colors) > 0:
    print("=" * 60)
    print("COLOR ANALYSIS BY STYLE")
    print("=" * 60)
    print(df_colors.groupby('style')[['brightness', 'color_variance']].describe().round(2))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Brightness distribution
    df_colors.boxplot(column='brightness', by='style', ax=axes[0])
    axes[0].set_title('Average Brightness by Art Style')
    axes[0].set_xlabel('Art Style')
    axes[0].set_ylabel('Brightness (0-255)')
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)
    
    # Color variance distribution
    df_colors.boxplot(column='color_variance', by='style', ax=axes[1])
    axes[1].set_title('Color Variance by Art Style')
    axes[1].set_xlabel('Art Style')
    axes[1].set_ylabel('Color Variance')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.show()
else:
    print("No valid color data available.")

## Section 9: Prepare Data for Model Training

In [ ]:
# Install TensorFlow if not already installed (uncomment for Colab)
# !pip install tensorflow tensorflow-hub

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

# Image preprocessing parameters
IMG_SIZE = 224
BATCH_SIZE = 32
TRAIN_SPLIT = 0.8

# Prepare training data
def load_and_preprocess_images(file_paths, labels, img_size=IMG_SIZE):
    """Load and preprocess images"""
    images = []
    valid_labels = []
    
    for img_path, label in zip(file_paths, labels):
        try:
            img = Image.open(img_path).convert('RGB')
            img = img.resize((img_size, img_size))
            img_array = np.array(img) / 255.0  # Normalize to 0-1
            images.append(img_array)
            valid_labels.append(label)
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
    
    return np.array(images), np.array(valid_labels)

if len(df_images) > 0:
    # Encode labels
    label_encoder = LabelEncoder()
    encoded_labels = label_encoder.fit_transform(df_images['style'])
    
    # Load images
    print("Loading and preprocessing images...")
    X, y = load_and_preprocess_images(df_images['file_path'].values, encoded_labels)
    
    # One-hot encode labels
    y_categorical = to_categorical(y, num_classes=len(ART_STYLES))
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_categorical, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"Training set size: {X_train.shape}")
    print(f"Test set size: {X_test.shape}")
    print(f"Number of classes: {len(ART_STYLES)}")
    print(f"Class labels: {label_encoder.classes_}")
else:
    print("No images available for preprocessing.")

## Section 10: Train the Art Style Classification Model

In [ ]:
if len(df_images) > 0 and 'X_train' in locals():
    # Data augmentation
    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomBrightness(0.2),
    ], name="data_augmentation")
    
    # Build CNN model for art style classification
    model = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        augmentation,
        layers.Rescaling(1./127.5, offset=-1),  # Normalize to [-1, 1]
        
        # Convolutional blocks
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Flattening and Dense layers
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(len(ART_STYLES), activation='softmax')  # 8 art styles
    ], name='ArtStyleClassifier')
    
    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
    )
    
    print("Model Architecture:")
    model.summary()
    
    # Callbacks
    early_stopping = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )
    
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=0.00001,
        verbose=1
    )
    
    # Train model
    print("\nTraining model...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=50,
        batch_size=BATCH_SIZE,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )
    
    print("Model training completed!")
else:
    print("Data not available for training.")

## Section 11: Model Performance Evaluation

In [ ]:
if len(df_images) > 0 and 'history' in locals() and 'model' in locals():
    from sklearn.metrics import classification_report, confusion_matrix
    
    # Evaluate on test set
    test_loss, test_accuracy, test_precision, test_recall = model.evaluate(X_test, y_test, verbose=0)
    
    print("=" * 60)
    print("MODEL PERFORMANCE EVALUATION")
    print("=" * 60)
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test Precision: {test_precision:.4f}")
    print(f"Test Recall: {test_recall:.4f}")
    
    # Predictions
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_test_classes = np.argmax(y_test, axis=1)
    
    # Classification report
    print("\n" + "=" * 60)
    print("CLASSIFICATION REPORT")
    print("=" * 60)
    print(classification_report(
        y_test_classes, 
        y_pred_classes, 
        target_names=label_encoder.classes_,
        digits=4
    ))
    
    # Training history plots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Accuracy
    axes[0, 0].plot(history.history['accuracy'], label='Training Accuracy')
    axes[0, 0].plot(history.history['val_accuracy'], label='Validation Accuracy')
    axes[0, 0].set_title('Model Accuracy', fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Loss
    axes[0, 1].plot(history.history['loss'], label='Training Loss')
    axes[0, 1].plot(history.history['val_loss'], label='Validation Loss')
    axes[0, 1].set_title('Model Loss', fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Precision
    axes[1, 0].plot(history.history['precision'], label='Training Precision')
    axes[1, 0].plot(history.history['val_precision'], label='Validation Precision')
    axes[1, 0].set_title('Model Precision', fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Recall
    axes[1, 1].plot(history.history['recall'], label='Training Recall')
    axes[1, 1].plot(history.history['val_recall'], label='Validation Recall')
    axes[1, 1].set_title('Model Recall', fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Confusion Matrix
    cm = confusion_matrix(y_test_classes, y_pred_classes)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title('Confusion Matrix', fontweight='bold', fontsize=14)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    # Save model
    model.save('art_style_classifier.h5')
    print("\nModel saved as 'art_style_classifier.h5'")
else:
    print("Model training required before evaluation.")